In [9]:
import numpy as np
from usearch.index import Index, Matches
import h5py
import pandas as pd
import numpy as np

In [ ]:
dataset_dict = {
    "mnist": "./data/mnist-784-euclidean.hdf5",
    "sift": "./data/sift-128-euclidean.hdf5",
}

dataset = "sift"

def get_data():
    with h5py.File(dataset_dict[dataset], "r") as f:
        train = f["train"][()]
        return train

def get_test():
    with h5py.File(dataset_dict[dataset], "r") as f:
        test = f["test"][()]
        return test

def transform_data(data):
    df = pd.DataFrame(data)
    df.insert(0, "id", range(0, len(df)))  # Add unique ID column
    df["vec"] = df.apply(lambda x: x.values[1:].tolist(), axis=1)  # Exclude ID from vector
    return df[["id", "vec"]]


train = transform_data(get_data())

test = transform_data(get_test())

dimensionality = len(train['vec'].iloc[0])

def get_ground_truth():
    """Loads ground-truth nearest neighbors from HDF5 file"""
    with h5py.File(dataset_dict[dataset], "r") as f:
        return f["neighbors"][()]  # Nearest neighbor indices


ground_truth = get_ground_truth()




In [ ]:
index = Index(
    ndim=dimensionality, # Define the number of dimensions in input vectors
    metric='l2sq', # Choose 'l2sq', 'haversine' or other metric, default = 'ip'
    dtype='f32', # Quantize to 'f16' or 'i8' if needed, default = 'f32'
)



for i in range(0, len(train)):
    index.add(i, np.array(train['vec'][i]))

In [ ]:
import time
import numpy as np

# Compute recall with average search time measurement
def compute_recall():
    total_recall = 0
    total_search_time = 0  # Track total search time
    num_queries = 10
    k = 100

    min_time = float('inf')
    max_time= float('-inf')

    for i in range(num_queries):
        random_number = np.random.randint(0, test.shape[0])
        vec = test['vec'][random_number]

        # Measure search time
        start_time = time.time()
        retrieved_neighbors = index.search(np.array(vec), k).keys
        end_time = time.time()

        total_search_time += (end_time - start_time)

        if(total_search_time > max_time):
            max_time = total_search_time
        
        if(total_search_time < min_time):
            min_time = total_search_time

        # Get ground truth neighbors
        ground_truth_neighbors = ground_truth[random_number][:k]

        # Compute recall for this query
        retrieved_set = set(retrieved_neighbors)
        ground_truth_set = set(ground_truth_neighbors)

        intersection_size = len(retrieved_set & ground_truth_set)
        recall = intersection_size / k
        total_recall += recall

    avg_recall = total_recall / num_queries
    avg_search_time = total_search_time / num_queries  # Compute average search time

    print(f"Average search time per query: {avg_search_time:.6f} seconds")
    return avg_recall, avg_search_time, min_time, max_time

print("Average recall:", compute_recall())


Average search time per query: 0.000723 seconds
Average recall: (0.8870000000000001, 0.0007234811782836914, 0.0021331310272216797, 0.007234811782836914)


In [ ]:
import json
import os


def delete_random_entries(iteration, filename):
    
    total_entries = index.size

    num_to_delete = int(0.05 * total_entries)  # 5% of dataset
    
    print(f"Total entries: {total_entries}, Deleting: {num_to_delete}")
    
    # Select random IDs to delete
    rows_to_delete = train.sample(n=num_to_delete)

    
    # Step 3: Delete selected rows

    for row in rows_to_delete.values:
        index.remove(row[0])


    
    # Verify deletion
    new_total_entries = index.size
    print(f"Remaining entries after deletion: {new_total_entries}")
    


    # Step 5: Reinsert deleted vectors
    for row in rows_to_delete.values:
        index.add(row[0], np.array(row[1]))
    
    # Step 6: Recompute recall after reinsertion
    recall_after_reinsertion, time_after_reinserting, min_time, max_time = compute_recall()
    print(f"Recall@10 after reinsertion: {recall_after_reinsertion:.4f}")
    
    # Step 7: Log results to JSON
    results = {
        'iteration': iteration,
        'min_query_time': min_time,
        'max_query_time': max_time,
        'avarage_query_time': time_after_reinserting,
        'total_entries': total_entries,
        'num_deleted': num_to_delete,
        'remaining_entries': new_total_entries,
        'recall_after_reinsertion': recall_after_reinsertion
    }
    
    if os.path.exists(filename):
        with open(filename, 'r') as f:
            data = json.load(f)
    else:
        data = []
    
    data.append(results)
    
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)
    
    



In [ ]:
filename = "hnsw_recall_experiment_2.json"

for i in range(0, 1000):
    print("Iteration: ", i)
    delete_random_entries(i, filename)

Iteration:  0
Total entries: 1000000, Deleting: 50000
Remaining entries after deletion: 950000
Average search time per query: 0.000587 seconds
Recall@10 after reinsertion: 0.8900
Iteration:  1
Total entries: 1000000, Deleting: 50000
Remaining entries after deletion: 950000
Average search time per query: 0.000575 seconds
Recall@10 after reinsertion: 0.9030
Iteration:  2
Total entries: 1000000, Deleting: 50000
Remaining entries after deletion: 950000
Average search time per query: 0.000619 seconds
Recall@10 after reinsertion: 0.8940
Iteration:  3
Total entries: 1000000, Deleting: 50000
Remaining entries after deletion: 950000
Average search time per query: 0.000523 seconds
Recall@10 after reinsertion: 0.9180
Iteration:  4
Total entries: 1000000, Deleting: 50000
Remaining entries after deletion: 950000
Average search time per query: 0.000477 seconds
Recall@10 after reinsertion: 0.8940
Iteration:  5
Total entries: 1000000, Deleting: 50000
Remaining entries after deletion: 950000
Average se